In [ ]:
# ============================================================================
# TECH CHALLENGE - FASE 4 - DATA ANALYTICS
# MODELO PREDITIVO PARA CLASSIFICAÇÃO DO NÍVEL DE OBESIDADE
# ============================================================================
#
# Este código foi preparado para ser copiado e executado em uma célula do
# Google Colab. Ele realiza todas as principais etapas da pipeline de Machine
# Learning exigida no Tech Challenge:
#
# 1. Upload e leitura da base de dados;
# 2. Limpeza complementar e remoção de registros duplicados;
# 3. Feature engineering com a criação do IMC;
# 4. Separação das variáveis de entrada e da variável alvo;
# 5. Pré-processamento de variáveis numéricas e categóricas;
# 6. Separação dos dados entre treinamento e teste;
# 7. Comparação de diferentes algoritmos;
# 8. Validação cruzada estratificada;
# 9. Avaliação do melhor modelo no conjunto de teste;
# 10. Exibição da matriz de confusão e importância das variáveis;
# 11. Salvamento do modelo final para uso no Streamlit.
#
# Caso alguma biblioteca não esteja disponível no Colab, execute antes:
# !pip install -q openpyxl joblib seaborn
# ============================================================================

## 1. IMPORTAÇÃO DAS BIBLIOTECAS

In [ ]:
# Evita que avisos não críticos deixem a saída do notebook poluída.
import warnings
warnings.filterwarnings("ignore")

# Pandas é utilizado para ler e manipular a base de dados.
import pandas as pd

# NumPy é utilizado para identificar tipos numéricos e realizar cálculos.
import numpy as np

# Matplotlib e Seaborn são utilizados para construir os gráficos.
import matplotlib.pyplot as plt
import seaborn as sns

# Joblib é utilizado para salvar o modelo treinado em um arquivo .pkl.
import joblib

# Ferramenta do Google Colab para fazer upload e download de arquivos.
from google.colab import files

# Display apresenta tabelas de maneira mais organizada dentro do Colab.
from IPython.display import display

# Clone cria uma cópia limpa do melhor pipeline antes do treinamento para deploy.
from sklearn.base import clone

# Funções usadas para separar a base e executar a validação cruzada.
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)

# ColumnTransformer permite aplicar tratamentos diferentes em grupos de colunas.
from sklearn.compose import ColumnTransformer

# Pipeline organiza o pré-processamento e o modelo em uma única estrutura.
from sklearn.pipeline import Pipeline

# OneHotEncoder converte textos em números e StandardScaler padroniza escalas.
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# SimpleImputer preenche possíveis valores ausentes.
from sklearn.impute import SimpleImputer

# O DummyClassifier funciona como uma referência mínima de desempenho.
from sklearn.dummy import DummyClassifier

# Algoritmo de classificação baseado em regressão logística.
from sklearn.linear_model import LogisticRegression

# Algoritmo de árvore de decisão.
from sklearn.tree import DecisionTreeClassifier

# Algoritmos de conjunto baseados em várias árvores de decisão.
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)

# Métricas utilizadas para avaliar o modelo final.
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Define um estilo visual padrão para os gráficos.
sns.set_theme(style="whitegrid")

## 2. UPLOAD E LEITURA DA BASE DE DADOS

In [ ]:
# Solicita ao usuário o envio da planilha utilizada no Tech Challenge.
print("Envie a planilha Obesity Análise Preliminar.xlsx")

# Abre a janela de upload do Google Colab.
arquivos = files.upload()

# Obtém o nome do primeiro arquivo que foi enviado.
nome_arquivo = list(arquivos.keys())[0]

# Verifica se o arquivo enviado é um Excel.
if nome_arquivo.lower().endswith((".xlsx", ".xls")):

    # Abre o arquivo para verificar quais abas existem.
    excel = pd.ExcelFile(nome_arquivo)

    # Na planilha do projeto, os dados estão na aba chamada Base_Data.
    if "Base_Data" in excel.sheet_names:
        nome_aba = "Base_Data"

    # Caso a aba Base_Data não exista, utiliza a primeira aba encontrada.
    else:
        nome_aba = excel.sheet_names[0]

    # Lê a aba selecionada e armazena os dados no DataFrame df.
    df = pd.read_excel(
        nome_arquivo,
        sheet_name=nome_aba
    )

# Caso o arquivo seja CSV, tenta identificar automaticamente o separador.
elif nome_arquivo.lower().endswith(".csv"):

    try:
        # Primeira tentativa utilizando a codificação UTF-8.
        df = pd.read_csv(
            nome_arquivo,
            sep=None,
            engine="python",
            encoding="utf-8"
        )

    except UnicodeDecodeError:
        # Segunda tentativa caso o arquivo utilize a codificação Latin-1.
        df = pd.read_csv(
            nome_arquivo,
            sep=None,
            engine="python",
            encoding="latin-1"
        )

# Interrompe a execução se o formato enviado não for Excel nem CSV.
else:
    raise ValueError("Envie um arquivo Excel ou CSV.")

# Remove possíveis espaços extras no início ou no final dos nomes das colunas.
df.columns = df.columns.astype(str).str.strip()

# Apresenta informações básicas para confirmar que a base foi carregada.
print("\nBase carregada com sucesso!")
print(f"Quantidade de linhas: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

# Mostra as cinco primeiras linhas da base.
display(df.head())

## 3. IDENTIFICAÇÃO DAS COLUNAS PRINCIPAIS

In [ ]:
# A mesma base pode apresentar diferentes nomes para a variável alvo.
possiveis_alvos = [
    "Obesity",
    "Obesity_level",
    "Obesity Level",
    "NObeyesdad",
    "Nível de Obesidade"
]

# Possíveis nomes para a coluna que contém a altura da pessoa.
possiveis_alturas = [
    "Altura (m)",
    "Height",
    "Altura"
]

# Possíveis nomes para a coluna que contém o peso da pessoa.
possiveis_pesos = [
    "Peso (Kg)",
    "Weight",
    "Peso"
]

# Procura na base o primeiro nome correspondente à variável alvo.
coluna_alvo = next(
    (
        coluna
        for coluna in possiveis_alvos
        if coluna in df.columns
    ),
    None
)

# Procura na base o primeiro nome correspondente à altura.
coluna_altura = next(
    (
        coluna
        for coluna in possiveis_alturas
        if coluna in df.columns
    ),
    None
)

# Procura na base o primeiro nome correspondente ao peso.
coluna_peso = next(
    (
        coluna
        for coluna in possiveis_pesos
        if coluna in df.columns
    ),
    None
)

# Interrompe o código se nenhuma coluna alvo conhecida for encontrada.
if coluna_alvo is None:
    raise ValueError(
        "A coluna alvo não foi encontrada. "
        f"Colunas existentes: {df.columns.tolist()}"
    )

# Interrompe o código se peso ou altura não forem encontrados.
if coluna_altura is None or coluna_peso is None:
    raise ValueError(
        "As colunas de peso ou altura não foram encontradas."
    )

# Informa quais colunas foram identificadas.
print(f"\nVariável alvo: {coluna_alvo}")
print(f"Coluna de altura: {coluna_altura}")
print(f"Coluna de peso: {coluna_peso}")

## 4. LIMPEZA COMPLEMENTAR DA BASE

In [ ]:
# Conta quantos registros completamente duplicados existem.
quantidade_duplicadas = df.duplicated().sum()

# Exibe a quantidade encontrada antes da remoção.
print(f"\nLinhas duplicadas encontradas: {quantidade_duplicadas}")

# Remove as linhas duplicadas para evitar que o mesmo registro apareça
# simultaneamente no conjunto de treinamento e no conjunto de teste.
df = df.drop_duplicates().reset_index(drop=True)

# Exibe a quantidade de registros após a limpeza.
print(f"Linhas após remover duplicadas: {len(df)}")

# Conta e exibe o total de valores ausentes existentes em toda a base.
print(f"Total de valores ausentes: {df.isna().sum().sum()}")

## 5. FEATURE ENGINEERING

In [ ]:
# Cria uma nova característica chamada IMC.
# Fórmula: IMC = peso dividido pela altura ao quadrado.
df["IMC"] = df[coluna_peso] / (df[coluna_altura] ** 2)

# Informa que a nova variável foi criada.
print("\nFeature engineering realizada: criação da variável IMC.")

# Mostra alguns exemplos do peso, altura, IMC e nível de obesidade.
display(
    df[
        [coluna_altura, coluna_peso, "IMC", coluna_alvo]
    ].head()
)

## 6. SEPARAÇÃO ENTRE VARIÁVEIS DE ENTRADA E VARIÁVEL ALVO

In [ ]:
# X recebe todas as colunas utilizadas para fazer a previsão.
# A coluna alvo é removida porque ela representa a resposta esperada.
X = df.drop(columns=[coluna_alvo])

# y recebe somente a coluna que o modelo deverá aprender a prever.
y = df[coluna_alvo]

# Calcula a quantidade e o percentual de registros de cada classe.
distribuicao = pd.DataFrame({
    "Quantidade": y.value_counts(),
    "Percentual": y.value_counts(normalize=True) * 100
})

# Exibe a distribuição da variável alvo.
print("\nDistribuição da variável alvo:")
display(distribuicao.round(2))

## 7. IDENTIFICAÇÃO DAS VARIÁVEIS NUMÉRICAS E CATEGÓRICAS

In [ ]:
# Seleciona as colunas que possuem valores numéricos.
colunas_numericas = X.select_dtypes(
    include=np.number
).columns.tolist()

# Seleciona as colunas que possuem textos ou categorias.
colunas_categoricas = X.select_dtypes(
    exclude=np.number
).columns.tolist()

# Exibe as listas para facilitar a conferência.
print("\nVariáveis numéricas:")
print(colunas_numericas)

print("\nVariáveis categóricas:")
print(colunas_categoricas)

## 8. PRÉ-PROCESSAMENTO DAS VARIÁVEIS NUMÉRICAS

In [ ]:
# Cria uma pipeline específica para as variáveis numéricas.
pipeline_numerica = Pipeline(
    steps=[
        (
            # Se houver um número ausente, ele será substituído pela mediana.
            "preenchimento",
            SimpleImputer(strategy="median")
        ),
        (
            # Padroniza os números para escalas comparáveis.
            "padronizacao",
            StandardScaler()
        )
    ]
)

## 9. PRÉ-PROCESSAMENTO DAS VARIÁVEIS CATEGÓRICAS

In [ ]:
# Cria uma pipeline específica para as variáveis de texto/categoria.
pipeline_categorica = Pipeline(
    steps=[
        (
            # Se houver uma categoria ausente, utiliza a categoria mais comum.
            "preenchimento",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            # Converte textos como Sim/Não em colunas numéricas binárias.
            "onehot",
            OneHotEncoder(
                # Categorias novas serão ignoradas, evitando erros no deploy.
                handle_unknown="ignore",

                # Retorna uma matriz comum, facilitando análises posteriores.
                sparse_output=False
            )
        )
    ]
)

## 10. UNIÃO DOS DOIS TIPOS DE PRÉ-PROCESSAMENTO

In [ ]:
# O ColumnTransformer aplica a pipeline numérica nas colunas numéricas e a
# pipeline categórica nas colunas de texto.
preprocessador = ColumnTransformer(
    transformers=[
        (
            "numericas",
            pipeline_numerica,
            colunas_numericas
        ),
        (
            "categoricas",
            pipeline_categorica,
            colunas_categoricas
        )
    ]
)

## 11. SEPARAÇÃO DOS DADOS ENTRE TREINAMENTO E TESTE

In [ ]:
# Separa 80% dos registros para treinamento e 20% para teste.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,

    # Reserva 20% da base para a avaliação final.
    test_size=0.20,

    # Garante que a divisão seja igual em todas as execuções.
    random_state=42,

    # Mantém a proporção das sete classes nos dois conjuntos.
    stratify=y
)

# Informa quantos registros foram utilizados em cada conjunto.
print("\nDivisão dos dados:")
print(f"Treinamento: {len(X_train)} registros")
print(f"Teste: {len(X_test)} registros")

## 12. DEFINIÇÃO DOS MODELOS QUE SERÃO COMPARADOS

In [ ]:
# Cada item do dicionário representa um algoritmo que será testado.
modelos = {

    # Modelo de referência: sempre prevê a classe mais frequente.
    "Dummy Classifier": DummyClassifier(
        strategy="most_frequent"
    ),

    # Modelo linear utilizado como uma alternativa mais simples.
    "Regressão Logística": LogisticRegression(
        max_iter=4000,
        random_state=42
    ),

    # Modelo que aprende regras de decisão a partir dos dados.
    "Árvore de Decisão": DecisionTreeClassifier(
        max_depth=12,
        class_weight="balanced",
        random_state=42
    ),

    # Modelo composto por várias árvores de decisão.
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    # Modelo semelhante ao Random Forest, mas com maior aleatoriedade.
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=400,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

## 13. CONFIGURAÇÃO DA VALIDAÇÃO CRUZADA

In [ ]:
# Divide os dados de treinamento em cinco partes estratificadas.
validacao = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Lista que armazenará as métricas médias de cada modelo.
resultados = []

# Dicionário que armazenará cada pipeline construída.
pipelines = {}

# Mensagem apresentada antes do início do processamento.
print("\nTreinando e comparando os modelos...\n")

## 14. TREINAMENTO E COMPARAÇÃO DOS MODELOS

In [ ]:
# Percorre cada algoritmo existente no dicionário de modelos.
for nome_modelo, algoritmo in modelos.items():

    # Une o pré-processamento e o algoritmo em uma única pipeline.
    pipeline_modelo = Pipeline(
        steps=[
            ("preprocessador", preprocessador),
            ("modelo", algoritmo)
        ]
    )

    # Guarda a pipeline para que ela possa ser recuperada depois.
    pipelines[nome_modelo] = pipeline_modelo

    # Executa a validação cruzada utilizando somente os dados de treinamento.
    metricas_validacao = cross_validate(
        pipeline_modelo,
        X_train,
        y_train,
        cv=validacao,
        scoring={
            "acuracia": "accuracy",
            "f1_macro": "f1_macro",
            "precisao_macro": "precision_macro",
            "recall_macro": "recall_macro"
        },
        n_jobs=-1
    )

    # Calcula a média das cinco rodadas e adiciona o resultado à lista.
    resultados.append({
        "Modelo": nome_modelo,
        "Acurácia CV": metricas_validacao[
            "test_acuracia"
        ].mean(),
        "Precisão macro CV": metricas_validacao[
            "test_precisao_macro"
        ].mean(),
        "Recall macro CV": metricas_validacao[
            "test_recall_macro"
        ].mean(),
        "F1 macro CV": metricas_validacao[
            "test_f1_macro"
        ].mean()
    })

## 15. TABELA COMPARATIVA DOS MODELOS

In [ ]:
# Converte os resultados em uma tabela do Pandas.
tabela_resultados = pd.DataFrame(resultados)

# Ordena os modelos do maior para o menor F1-score macro.
tabela_resultados = tabela_resultados.sort_values(
    by="F1 macro CV",
    ascending=False
).reset_index(drop=True)

# Exibe a tabela de resultados em formato percentual.
print("Comparação dos modelos na validação cruzada:")
display(
    tabela_resultados.style.format({
        "Acurácia CV": "{:.2%}",
        "Precisão macro CV": "{:.2%}",
        "Recall macro CV": "{:.2%}",
        "F1 macro CV": "{:.2%}"
    })
)

## 16. GRÁFICO DE COMPARAÇÃO DOS MODELOS

In [ ]:
# Cria a área do gráfico.
plt.figure(figsize=(10, 5))

# Constrói um gráfico de barras com o F1-score de cada modelo.
sns.barplot(
    data=tabela_resultados,
    x="F1 macro CV",
    y="Modelo",
    palette="Blues_r"
)

# Desenha uma linha vermelha indicando a meta mínima de 75%.
plt.axvline(
    0.75,
    color="red",
    linestyle="--",
    label="Meta mínima de 75%"
)

# Configura o título, os eixos e a legenda do gráfico.
plt.title("Comparação dos modelos")
plt.xlabel("F1-score macro médio")
plt.ylabel("Modelo")
plt.xlim(0, 1)
plt.legend()

# Exibe o gráfico.
plt.show()

## 17. ESCOLHA E TREINAMENTO DO MELHOR MODELO

In [ ]:
# Seleciona o primeiro modelo da tabela, ou seja, o maior F1-score macro.
nome_melhor_modelo = tabela_resultados.loc[
    0,
    "Modelo"
]

# Recupera a pipeline correspondente ao modelo selecionado.
melhor_pipeline = pipelines[nome_melhor_modelo]

# Treina o melhor pipeline utilizando os 80% destinados ao treinamento.
melhor_pipeline.fit(
    X_train,
    y_train
)

# Faz previsões sobre os 20% que o modelo ainda não viu.
previsoes = melhor_pipeline.predict(X_test)

# Informa o nome do algoritmo selecionado.
print(f"\nMelhor modelo: {nome_melhor_modelo}")

## 18. CÁLCULO DAS MÉTRICAS FINAIS NO CONJUNTO DE TESTE

In [ ]:
# Acurácia: percentual total de previsões corretas.
acuracia = accuracy_score(
    y_test,
    previsoes
)

# Precisão macro: média da precisão calculada separadamente para cada classe.
precisao = precision_score(
    y_test,
    previsoes,
    average="macro"
)

# Recall macro: média da capacidade de encontrar cada uma das classes.
recall = recall_score(
    y_test,
    previsoes,
    average="macro"
)

# F1-score macro: equilíbrio médio entre precisão e recall nas sete classes.
f1 = f1_score(
    y_test,
    previsoes,
    average="macro"
)

# Apresenta todas as métricas em formato percentual.
print("\nMétricas finais:")
print(f"Acurácia:       {acuracia:.2%}")
print(f"Precisão macro: {precisao:.2%}")
print(f"Recall macro:   {recall:.2%}")
print(f"F1-score macro: {f1:.2%}")

# Verifica se a acurácia atende à exigência estabelecida no Tech Challenge.
if acuracia >= 0.75:
    print(
        "\nO modelo atingiu a exigência de "
        "acurácia superior a 75%."
    )
else:
    print(
        "\nO modelo ainda não atingiu a "
        "exigência de acurácia superior a 75%."
    )

## 19. RELATÓRIO DETALHADO DE CLASSIFICAÇÃO

In [ ]:
# Exibe precisão, recall e F1-score individualmente para cada classe.
print("\nRelatório de classificação:\n")
print(
    classification_report(
        y_test,
        previsoes,
        digits=4
    )
)

## 20. MATRIZ DE CONFUSÃO

In [ ]:
# Recupera a ordem das classes conhecida pelo modelo treinado.
classes = melhor_pipeline.named_steps[
    "modelo"
].classes_

# Calcula a matriz comparando as classes reais com as classes previstas.
matriz = confusion_matrix(
    y_test,
    previsoes,
    labels=classes
)

# Define o tamanho do gráfico.
plt.figure(figsize=(12, 9))

# Constrói o mapa de calor da matriz de confusão.
sns.heatmap(
    matriz,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)

# Configura o título e os nomes dos eixos.
plt.title(f"Matriz de confusão - {nome_melhor_modelo}")
plt.xlabel("Classe prevista")
plt.ylabel("Classe verdadeira")

# Ajusta a rotação dos nomes para melhorar a leitura.
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

# Exibe a matriz de confusão.
plt.show()

## 21. IMPORTÂNCIA DAS VARIÁVEIS

In [ ]:
# Recupera somente o algoritmo treinado que está dentro da pipeline.
modelo_treinado = melhor_pipeline.named_steps["modelo"]

# Recupera os nomes das variáveis após o One-Hot Encoding.
nomes_variaveis = melhor_pipeline.named_steps[
    "preprocessador"
].get_feature_names_out()

# Modelos baseados em árvores possuem o atributo feature_importances_.
if hasattr(modelo_treinado, "feature_importances_"):
    valores_importancia = modelo_treinado.feature_importances_

# Modelos lineares possuem coeficientes em vez de feature_importances_.
elif hasattr(modelo_treinado, "coef_"):
    valores_importancia = np.mean(
        np.abs(modelo_treinado.coef_),
        axis=0
    )

# Se o modelo não disponibilizar nenhuma das duas opções, não gera o gráfico.
else:
    valores_importancia = None

# Continua somente se as importâncias estiverem disponíveis.
if valores_importancia is not None:

    # Cria uma tabela associando cada variável ao seu nível de importância.
    importancia = pd.DataFrame({
        "Variável": nomes_variaveis,
        "Importância": valores_importancia
    })

    # Ordena e seleciona as quinze variáveis mais importantes.
    importancia = importancia.sort_values(
        by="Importância",
        ascending=False
    ).head(15)

    # Exibe a tabela.
    print("\nVariáveis mais importantes:")
    display(importancia)

    # Define o tamanho do gráfico.
    plt.figure(figsize=(10, 7))

    # Cria o gráfico de barras das importâncias.
    sns.barplot(
        data=importancia,
        x="Importância",
        y="Variável",
        palette="viridis"
    )

    # Configura o título e os nomes dos eixos.
    plt.title("15 variáveis mais importantes para o modelo")
    plt.xlabel("Importância")
    plt.ylabel("Variável")
    plt.tight_layout()

    # Exibe o gráfico.
    plt.show()

## 22. TREINAMENTO DO MODELO PARA O DEPLOY

In [ ]:
# Cria uma cópia limpa do melhor pipeline.
modelo_deploy = clone(melhor_pipeline)

# Depois de comprovar o desempenho, treina a versão de deploy com toda a base.
modelo_deploy.fit(
    X,
    y
)

## 23. SALVAMENTO DOS RESULTADOS

In [ ]:
# Salva a pipeline completa em um arquivo que será usado no Streamlit.
joblib.dump(
    modelo_deploy,
    "modelo_obesidade.pkl"
)

# Salva a tabela de comparação em CSV para documentação do trabalho.
tabela_resultados.to_csv(
    "comparacao_modelos.csv",
    index=False
)

# Informa os arquivos gerados.
print("\nArquivos criados:")
print("- modelo_obesidade.pkl")
print("- comparacao_modelos.csv")

# Como o IMC foi criado antes da pipeline, a aplicação Streamlit também deverá
# calculá-lo com a mesma fórmula antes de enviar os dados para o modelo.
print(
    "\nA aplicação Streamlit deverá calcular o IMC antes "
    "de enviar os dados ao modelo."
)

# Inicia o download do arquivo contendo o modelo treinado.
files.download("modelo_obesidade.pkl")

# Mensagem final solicitada para deixar o resultado da acurácia bem evidente.
print(f"\nO modelo obteve {acuracia * 100:.2f}% de acurácia.")